<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/02_backends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Files as the agent's workspace

In lesson 01 your agent wrote `report.md`. Where did that file go?

It turns out that question — **what outlives this conversation?** — is the most consequential
design decision in an agent, and it has a name: the **backend**.

**New in this lesson:** `StateBackend`, `StoreBackend`, `FilesystemBackend`, and `CompositeBackend`

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-02-backends"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Why an agent needs a filesystem at all

Here is the problem:

Every tool result stays in the message history and is **re-sent on every subsequent model
call**. A 20,000-token search result is not a one-time cost — it is 20,000 tokens per step, for
the rest of the run.

A filesystem allows the agent to **offload context** to a file so that it does not bloat the context window. The agent and retrieve (`grep`) the file contents on an as needed basis.

In Deep Agents, we use a concept of a **backend** that can represent a real or virtual filesystem. We will look at three built-in options.

---

## 2. The default: `StateBackend`

Files live in the agent's **graph state** — the same place its message list lives. Fast,
requires no setup, and lasts exactly as long as the conversation thread does.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant. Save any notes you take as files.",
)
agent

In [ ]:
from langsmith_studio_nb import start_studio

start_studio("agent")

Example prompt:
> Write a two-line haiku about debugging and save it to haiku.md

---

## 3. Persisting across threads: `StoreBackend`

A `Store` is a key-value store that lives outside any single conversation. Point the backend at
one and files survive from thread to thread.

In [ ]:
from deepagents.backends import StoreBackend

store_backend_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant. Save any notes you take as files.",
    # A namespace keeps one agent's files from colliding with another's.
    backend=StoreBackend(namespace=lambda rt: ("workshop", "files")),
)
store_backend_agent

In [ ]:
start_studio("store_backend_agent")

Example prompt:
> Write a two-line haiku about debugging and save it to haiku.md

---

## 4. Real files on a real disk: `FilesystemBackend`

Sometimes you want files you can actually look at — and that other programs can read.

In [ ]:
from deepagents.backends import FilesystemBackend

disk_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant. Save any notes you take as files.",
    backend=FilesystemBackend(root_dir="workspace"),
)
disk_agent

In [ ]:
start_studio("disk_agent")

Example prompt:
> Write a two-line haiku about debugging and save it to haiku.md

In [ ]:
# These are ordinary files. Inspect them with ordinary tools.
!ls -la workspace
!cat workspace/haiku.md

---

## 5. `CompositeBackend`: different paths, different homes

Route by path prefix. Here `/memories/` persists forever while everything else stays
thread-local scratch.

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend

routed_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a helpful assistant.\n"
        "Durable facts about the user go in /memories/.\n"
        "Everything else is scratch work."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(namespace=lambda rt: ("workshop", "memories"))},
    ),
)
routed_agent

In [ ]:
start_studio("routed_agent")

Example prompts:
> Remember that I prefer metric units and save it to /memories/preferences.md. Also jot a throwaway note to scratch.md.

> What do you know about my preferences? Also, does scratch.md still exist?

---

## 📌 Key takeaways

- The filesystem is a **context-management** tool: it keeps bulky content out of the message history so you pay for it once.
- "Which backend?" is really the question **"what should outlive this thread?"**
- `StateBackend` is thread-scoped, `StoreBackend` crosses threads, `FilesystemBackend` puts real files on a disk.
- `CompositeBackend` routes by path prefix, so one agent can have both scratch space and long-term memory.
- Explore our backend integrations to use S3, Azure Blob, PostgreSQL, and more as virtual filesystem backends.

---

## ➡️ Next

**[03 · Tools an agent can actually use](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/03_tools.ipynb)**

You have seen the tools that come free. Next you write your own — for a customer support agent —
and find out why a tool's **docstring** is one of the highest-leverage prompts in your codebase.